In [1]:
%%capture --no-stderr
%pip install --quiet -U langchain_core langgraph langchain_openai

In [2]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("PPLX_API_KEY")

PPLX_API_KEY:  ········


In [3]:
_set_env("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langchain-academy"

LANGSMITH_API_KEY:  ········


In [ ]:
from langchain_openai import ChatPerplexity
model = ChatPerplexity(model="sonar-pro",temperature=0)

In [6]:
from langgraph.graph import MessagesState
class State(MessagesState):
    running_outline: str

In [45]:
from langchain_core.messages import SystemMessage, HumanMessage, RemoveMessage

def call_model(state: State):
    
    running_outline = state.get("code_summary", "")

    if code_summary:
        
        system_message = f"Conversation outline so far: {code_summary}"

        messages = [SystemMessage(content=system_message)] + state["messages"]
    
    else:
        messages = state["messages"]
    
    response = model.invoke(messages)
    return {"messages": response}

In [2]:
def summarize_conversation(state: State):
    
    running_outline = state.get("code_summary", "")

    if running_outline:
        
        outline_message = (
            f"This is the running outline so far: {code_summary}\n\n"
            "create a summary:"
        )
        
    else:
        outline_message = "Create a running outline of the conversation above:"

    messages = state["messages"] + [HumanMessage(content=code_message)]
    response = model.invoke(messages)
    
    delete_messages = [RemoveMessage(id=m.id) for m in state["messages"][:-2]]
    return {"running_outline": response.content, "messages": delete_messages}

NameError: name 'State' is not defined

In [1]:
from langgraph.graph import END
from typing_extensions import Literal
def should_continue(state: State) -> Literal ["summarize_code",END]:
    
    """Return the next node to execute."""
    
    messages = state["messages"]
    
    if len(messages) > 6:
        return "summarize_code"
    
    return END

NameError: name 'State' is not defined

In [ ]:
from IPython.display import Image, display
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START

workflow = StateGraph(State)
workflow.add_node("conversation", call_model)
workflow.add_node(summarize_code)

workflow.add_edge(START, "code")
workflow.add_conditional_edges("conversation", should_continue)
workflow.add_edge("summarize_code", END)

memory = MemorySaver()
graph = workflow.compile(checkpointer=memory)
display(Image(graph.get_graph().draw_mermaid_png()))

## Threads

The checkpointer saves the state at each step as a checkpoint.

These saved checkpoints can be grouped into a `thread` of conversation.

Think about Slack as an analog: different channels carry different conversations.

Threads are like Slack channels, capturing grouped collections of state (e.g., conversation).

Below, we use `configurable` to set a thread ID.

![state.jpg](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbadf3b379c2ee621adfd1_chatbot-summarization1.png)

In [51]:
graph.get_state(config).values.get("summarize_code","")

''